## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key=os.getenv("GROQ_API_KEY")


In [2]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000028923313FD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000028923338550>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hii, my name is himanshu and I am a Chief AI engineer")])

AIMessage(content="Hello Himanshu, it's great to meet you. Being a Chief AI Engineer is a fascinating role, and I'm sure you have a wealth of knowledge and experience in the field of Artificial Intelligence. What specific areas of AI are you most interested in or have expertise in, such as machine learning, natural language processing, computer vision, or something else?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 51, 'total_tokens': 124, 'completion_time': 0.203854029, 'completion_tokens_details': None, 'prompt_time': 0.004092216, 'prompt_tokens_details': None, 'queue_time': 0.054786164, 'total_time': 0.207946245}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3f76-8402-7ad2-9757-d4b5f868daf4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 51, 'output_tokens':

In [ ]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hii, my name is himanshu and I am a Chief AI engineer"),
        AIMessage(content="Nice to meet you, Himanshu. As a Chief AI Engineer, I'm sure you have a wealth of knowledge and experience in the field of Artificial Intelligence. What specific areas of AI are you interested in or currently working on? Are you exploring advancements in Machine Learning, Natural Language Processing, Computer Vision, or perhaps something else?"),
        HumanMessage(content="Hey what's my name and what do i do?")
    ]
)


AIMessage(content='Your name is Himanshu, and you are a Chief AI Engineer.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 139, 'total_tokens': 155, 'completion_time': 0.022048301, 'completion_tokens_details': None, 'prompt_time': 0.01057636, 'prompt_tokens_details': None, 'queue_time': 0.05378169, 'total_time': 0.032624661}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019de9ce-9644-7c40-9187-a23766c91a1f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 139, 'output_tokens': 16, 'total_tokens': 155})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [ ]:
config = {"configurable":{'session_id':"chat1"}}

In [ ]:
response=with_message_history.invoke(
    [HumanMessage(content="Hii, my name is himanshu and I am a Chief AI engineer")],
    config=config
)

In [ ]:
response.content

'It seems like you already said this earlier. Anyway, as a Chief AI Engineer, what kind of projects have you worked on or are working on currently?'

In [ ]:
with_message_history.invoke(
    [HumanMessage(content="what's my name?")],
    config=config
)

AIMessage(content="Your name is Himanshu. You're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 211, 'total_tokens': 226, 'completion_time': 0.015260183, 'completion_tokens_details': None, 'prompt_time': 0.011987631, 'prompt_tokens_details': None, 'queue_time': 0.052508149, 'total_time': 0.027247814}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019de9d2-4ab0-7360-87a0-3ab8e58ceffd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 211, 'output_tokens': 15, 'total_tokens': 226})

In [ ]:
## change the config -->session ID
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="what's my name")],
    config=config1
)

response.content

"I don't have any information about your name. This is the beginning of our conversation, and I don't retain any data about individual users. If you'd like, you can tell me your name, and I'll be happy to chat with you!"

In [ ]:
response=with_message_history.invoke(
    [HumanMessage(content="my name is Abhishek")],
    config=config1
)

response.content

'Hello Abhishek. It was nice meeting you. Is there something I can help you with or would you like to chat about a particular topic?'

In [ ]:
response=with_message_history.invoke(
    [HumanMessage(content="what's my name?")],
    config=config1
)

response.content

'Your name is Abhishek.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are  a helpful assistance. Answer all the question to the most of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [ ]:
chain.invoke({"messages":[HumanMessage(content="hii my name is Himanshu")]})

AIMessage(content="Nice to meet you, Himanshu. I'm happy to assist you with any questions or topics you'd like to discuss. How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 60, 'total_tokens': 95, 'completion_time': 0.049435765, 'completion_tokens_details': None, 'prompt_time': 0.00404355, 'prompt_tokens_details': None, 'queue_time': 0.051044769, 'total_time': 0.053479315}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019de9e4-2e7f-7ff1-b25d-dd170ae8846c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 35, 'total_tokens': 95})

In [ ]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)


In [ ]:
config2={"configurable":{"session_id":"chat3"}}

In [ ]:
response=with_message_history.invoke(
    [HumanMessage(content="hii my name is Himanshu")],
    config=config2
)

In [ ]:
response.content

'Nice to meet you, Himanshu! How are you doing today? Is there anything I can help you with, or would you like to chat?'

In [ ]:
## Add more Complexity

prompt = ChatPromptTemplate.from_messages(
    [
        ("system","you are a helpful assistance. Answer all question to the best of your ability in{language}."),
        MessagesPlaceholder(variable_name="messages")
    ]
)
chain=prompt | model

In [ ]:
response=chain.invoke({"messages":[HumanMessage(content="Hi my name is Abhi")],"language":"Hindi"})
response.content

'नमस्ते Abhi जी, मैं आपकी मदद करने के लिए यहाँ हूँ। क्या आप कुछ पूछना चाहते हैं या मुझसे कोई जानकारी चाहते हैं?'

In [ ]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [ ]:
config4 = {'configurable':{"session_id":"chat4"}}
response=with_message_history.invoke(
    {"messages": [HumanMessage(content="Hii my name is Jadu")],"language":"Hindi"},
    config=config4
)
response.content

'नमस्कार Jadu! मैं आपकी मदद करने के लिए यहाँ हूँ। आपको किस बारे में पूछना है?'

In [ ]:
response=with_message_history.invoke(
    {"messages": [HumanMessage(content="what's my name?")],"language":"Hindi"},
    config=config4
)

response.content

'आपका नाम जादू (Jadu) है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [ ]:
from langchain_core.messages import SystemMessage,trim_messages

trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
response=trimmer.invoke(messages)

In [ ]:
response

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [ ]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough
chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |prompt
    |model
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="what ice cream do i like")],
    "language":"English"
    }
)
response.content

"I don't have any information about your preferences. You didn't mention what flavor you like before. Would you like to tell me?"

In [ ]:
response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="what math problem did i ask")],
    "language":"English"
    }
)
response.content

'Your math problem was 2 + 2.'

In [ ]:
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config1={"configurable":{"session_id":"chat5"}}

In [ ]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config1,
)

response.content

"You didn't mention your name. I'm happy to chat with you, but I don't have any information about your name."

In [ ]:
response=with_message_history.invoke(
    {
    "messages":messages + [HumanMessage(content="what math problem did i ask")],
    "language":"English"
    },
    config=config1
)
response.content

'You asked the math problem "2 + 2".'